# Psychotherapeutische Versorgung in Nordbayern – Datenbasis

Dieses Notebook bereitet zentrale INKAR-Strukturdaten für Nordbayern auf.
Es dient als Grundlage für spätere Nachfrage-, Angebots- und
Warteschlangen-Simulationen.

Verwendete Variablen:
- Bevölkerung insgesamt
- Bodenfläche
- Einwohnerdichte

Alle Analysen erfolgen auf Landkreisebene

In [14]:
# --- Robust Parquet Loader für Positron/Jupyter ---
from pathlib import Path

# 1) Autoreload (falls aktiv) programmgesteuert deaktivieren
try:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        # deaktiviert autoreload, ohne dass du Magics tippen musst
        ip.run_line_magic("autoreload", "0")
        try:
            ip.run_line_magic("unload_ext", "autoreload")
        except Exception:
            pass
except Exception:
    pass

# 2) Datei automatisch suchen (im Projektordner und Unterordnern)
filename = "inkar_bayern_nordbayern.parquet"
root = Path.cwd()

candidates = list(root.rglob(filename))
if not candidates:
    raise FileNotFoundError(
        f"Datei '{filename}' nicht gefunden unter: {root}\n"
        "Tipp: Lege die Datei in den gleichen Ordner wie das Notebook oder in einen 'data/'-Ordner."
    )

parquet_path = candidates[0]
print("Gefundene Datei:", parquet_path)

# 3) Parquet robust lesen: erst pyarrow -> dann in pandas DataFrame konvertieren
import pyarrow.parquet as pq

table = pq.read_table(parquet_path)  # liest Parquet als Arrow Table
print("Arrow Table:", table.num_rows, "Zeilen,", table.num_columns, "Spalten")

import pandas as pd
df = table.to_pandas()  # Konvertierung zu pandas

df.head()

The autoreload extension doesn't define how to unload it.


FileNotFoundError: Datei 'inkar_bayern_nordbayern.parquet' nicht gefunden unter: /Users/renatafigueroa/Dokumente/ohm/3.Semester/Datenvisualisierung /Projekt_Datenvisualisierung/notebooks
Tipp: Lege die Datei in den gleichen Ordner wie das Notebook oder in einen 'data/'-Ordner.

In [19]:
import os
print(os.listdir(".."))        # Projektroot
print(os.listdir("../data"))   # Datenordner

['.DS_Store', 'output', 'docs', 'README.md', '.gitignore', '.venv', '20251229_GitHub.ipynb', '.git', 'data', 'notebooks']
['inkar_load.ipynb', 'inkar_bayern_nordbayern.parquet', 'Psychotherapeuten.csv', '20251215.csv', 'readme.md', 'merged_nordbayern.csv', 'bayern_landkreise.geo.json']


In [21]:
import duckdb
import pandas as pd

DATA_PATH = "../data/inkar_bayern_nordbayern.parquet"

df = duckdb.query(
    f"SELECT * FROM read_parquet('{DATA_PATH}')"
).to_df()

df.head()

,Kennziffer,Name,Nordbayern,Raumbezug,Zeitbezug,Indikator,Kuerzel,Bereich,ID,Wert
0,09161000,Ingolstadt,False,Kreise,2023,Bevölkerung,a_bb_4G,EU,None,100.0
1,09161000,Ingolstadt,False,Kreise,2023,Bildung,a_bb_4G,EU,None,100.0
2,09161000,Ingolstadt,False,Kreise,2023,Gesundheit,a_bb_4G,EU,None,100.0
3,09161000,Ingolstadt,False,Kreise,2023,Siedlungsstruktur,a_bb_4G,EU,None,100.0
4,09161000,Ingolstadt,False,Kreise,2023,Verkehr,a_bb_4G,EU,None,100.0


In [22]:
indikatoren = [
    "Einwohnerdichte (Einwohner je km²)",
    "Bevölkerung insgesamt",
    "Bodenfläche"
]

df_sel = df[df["Indikator"].isin(indikatoren)].copy()
df_sel["Indikator"].value_counts()

Series([], Name: count, dtype: int64)

In [23]:
df_wide = (
    df_sel
    .pivot_table(
        index=["Kennziffer", "Name"],
        columns="Indikator",
        values="Wert",
        aggfunc="mean"
    )
    .reset_index()
)

df_wide = df_wide.rename(columns={
    "Bevölkerung insgesamt": "population",
    "Bodenfläche": "area_km2",
    "Einwohnerdichte (Einwohner je km²)": "population_density"
})

df_wide.head()

Indikator,Kennziffer,Name


In [28]:
df[df["Indikator"].str.contains("Bevölkerung", na=False)]["Indikator"].unique()

array(['Bevölkerung', 'Bevölkerung (mit BBSR-Zensuskorrekturen)',
       'Bevölkerung gesamt', 'Bevölkerung weiblich',
       'Bevölkerung männlich', 'Bevölkerungsentwicklung (5 Jahre)',
       'Schutzsuchende an Bevölkerung',
       'Schutzsuchende an ausländischer Bevölkerung',
       'Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre)',
       'Durchschnittsalter der Bevölkerung',
       'Bevölkerung in Mittelzentren', 'Bevölkerung in Oberzentren',
       'Bevölkerungsentwicklung (10 Jahre)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2030)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2035)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2040)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2045)',
       'Bevölkerung – Altersstruktur',
       'Bevölkerung – Bevölkerungsstruktur', 'Bevölkerung – Wanderungen',
       'Bevölkerung - Natürliche Bevölkerungsbewegungen',
       'Bevölkerungsentwicklung'], dtype=object)

In [29]:
df[df["Indikator"].str.contains("Einwohner", na=False)]["Indikator"].unique()

array(['Einwohner 65 Jahre und älter', 'Einwohner 75 Jahre und älter',
       'Erholungsfläche je Einwohner',
       'Bruttoinlandsprodukt je Einwohner', 'Einwohner unter 6 Jahre',
       'Einwohner von 6 bis unter 18 Jahren',
       'Baugenehmigungen für Wohnungenje Einwohner',
       'Einwohner von 18 bis unter 25 Jahren',
       'Einwohner von 25 bis unter 30 Jahren',
       'Einwohner von 30 bis unter 50 Jahren',
       'Einwohner von 50 bis unter 65 Jahren',
       'Einwohner 75 Jahre und älter, Frauen', 'Einwohner unter 3 Jahren',
       'Einwohner von 3 bis unter 6 Jahren',
       'Auszubildende je 100 Einwohner 15 bis 25 Jahre',
       'Studierende je 100 Einwohner 18 bis 25 Jahre',
       'Naturnähere Fläche je Einwohner', 'Freifläche je Einwohner',
       'Neubauwohnungen je Einwohner',
       'Neubauwohnungen in Ein- und Zweifamilienhäusern je Einwohner',
       'Neubauwohnungen in Mehrfamilienhäusern je Einwohner',
       'Einwohner von 18 bis unter 25 Jahren, Frauen',
    

In [30]:
df_sel = df[
    df["Indikator"].str.contains(
        "Bevölkerung|Einwohnerdichte|Bodenfläche",
        regex=True,
        na=False
    )
].copy()

df_sel["Indikator"].value_counts()

Indikator
Bevölkerung gesamt                                     3016
Bevölkerung weiblich                                   3016
Bevölkerung männlich                                   3016
Bevölkerungsentwicklung (5 Jahre)                      3016
Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre)      3016
Durchschnittsalter der Bevölkerung                     3016
Bevölkerungsentwicklung (10 Jahre)                     3016
Bevölkerung (mit BBSR-Zensuskorrekturen)               3016
Bodenfläche gesamt qkm                                 2655
Einwohnerdichte                                        2655
Schutzsuchende an Bevölkerung                          1768
Schutzsuchende an ausländischer Bevölkerung            1768
Bevölkerung                                             871
Bevölkerungsentwicklung                                 679
Bevölkerung in Mittelzentren                            520
Bevölkerung in Oberzentren                              520
Prognostizierte Bevölkerungsen

In [31]:
df_wide = (
    df_sel
    .pivot_table(
        index=["Kennziffer", "Name"],
        columns="Indikator",
        values="Wert",
        aggfunc="mean"
    )
    .reset_index()
)

df_wide.head()

Indikator,Kennziffer,Name,Bevölkerung,Bevölkerung (mit BBSR-Zensuskorrekturen),Bevölkerung - Natürliche Bevölkerungsbewegungen,Bevölkerung gesamt,Bevölkerung in Mittelzentren,Bevölkerung in Oberzentren,Bevölkerung männlich,Bevölkerung weiblich,Bevölkerung – Altersstruktur,Bevölkerung – Bevölkerungsstruktur,Bevölkerung – Wanderungen,Bevölkerungsentwicklung,Bevölkerungsentwicklung (10 Jahre),Bevölkerungsentwicklung (5 Jahre),Bodenfläche gesamt qkm,Durchschnittsalter der Bevölkerung,Einwohnerdichte,Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre),Prognostizierte Bevölkerungsentwicklung (2022-2030),Prognostizierte Bevölkerungsentwicklung (2022-2035),Prognostizierte Bevölkerungsentwicklung (2022-2040),Prognostizierte Bevölkerungsentwicklung (2022-2045),Schutzsuchende an Bevölkerung,Schutzsuchende an ausländischer Bevölkerung
0,09,Bayern,1.311393e+07,1.249043e+07,NaN,1.258761e+07,18.676,35.41,6.185950e+06,6.401664e+06,NaN,NaN,NaN,2.397143,4.763448,2.007241,70545.350385,41.907931,182.053846,8.378050e+06,2.48,3.30,3.96,4.40,1.199412,9.545294
1,091,Oberbayern,NaN,4.326012e+06,NaN,4.365552e+06,15.326,41.72,2.142116e+06,2.223436e+06,NaN,NaN,NaN,NaN,7.273448,3.370000,17529.595263,41.559655,256.008421,2.939161e+06,3.82,5.35,6.75,7.94,1.282941,7.759412
2,09161000,Ingolstadt,1.069538e+05,1.248367e+05,100.0,1.255357e+05,0.000,100.00,6.260279e+04,6.293293e+04,100.0,100.0,100.0,3.927143,10.610690,4.266552,133.356154,41.112414,988.433077,8.463045e+04,3.69,5.07,6.30,7.31,1.910588,10.028235
3,09162000,"München, Landeshauptstadt",1.151902e+06,1.335517e+06,100.0,1.343794e+06,0.000,100.00,6.509700e+05,6.928238e+05,100.0,100.0,100.0,2.995714,6.082069,3.214828,310.659615,41.343448,4572.682692,9.403802e+05,4.41,6.19,7.90,9.44,2.062353,8.171765
4,09163000,"Rosenheim, Stadt",4.948311e+04,6.011859e+04,100.0,6.091645e+04,0.000,100.00,2.982931e+04,3.108714e+04,100.0,100.0,100.0,3.024286,4.425172,1.778276,37.220385,42.003448,1670.357692,4.104459e+04,1.57,2.04,2.67,3.30,1.408824,6.966471


In [32]:
rename_map = {}

for c in df_wide.columns:
    if isinstance(c, str):
        if "Bevölkerung" in c and "gesamt" in c:
            rename_map[c] = "population"
        elif "Einwohnerdichte" in c:
            rename_map[c] = "population_density"
        elif "Bodenfläche" in c:
            rename_map[c] = "area_km2"

df_wide = df_wide.rename(columns=rename_map)

print("Umbenannt:", rename_map)
df_wide.columns

Umbenannt: {'Bevölkerung gesamt': 'population', 'Bodenfläche gesamt qkm': 'area_km2', 'Einwohnerdichte': 'population_density'}


Index(['Kennziffer', 'Name', 'Bevölkerung',
       'Bevölkerung (mit BBSR-Zensuskorrekturen)',
       'Bevölkerung - Natürliche Bevölkerungsbewegungen', 'population',
       'Bevölkerung in Mittelzentren', 'Bevölkerung in Oberzentren',
       'Bevölkerung männlich', 'Bevölkerung weiblich',
       'Bevölkerung – Altersstruktur', 'Bevölkerung – Bevölkerungsstruktur',
       'Bevölkerung – Wanderungen', 'Bevölkerungsentwicklung',
       'Bevölkerungsentwicklung (10 Jahre)',
       'Bevölkerungsentwicklung (5 Jahre)', 'area_km2',
       'Durchschnittsalter der Bevölkerung', 'population_density',
       'Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2030)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2035)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2040)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2045)',
       'Schutzsuchende an Bevölkerung',
       'Schutzsuchende an ausländischer Bevölkerung']

In [33]:
df_wide["demand_raw"] = (
    df_wide["population"] *
    df_wide["population_density"]
)

df_wide["demand_index"] = (
    (df_wide["demand_raw"] - df_wide["demand_raw"].mean()) /
    df_wide["demand_raw"].std()
)

df_wide[["Name", "demand_index"]].sort_values(
    "demand_index", ascending=False
).head(10)

Indikator,Name,demand_index
3,"München, Landeshauptstadt",9.101477
0,Bayern,3.246182
67,Nürnberg,1.850259
1,Oberbayern,1.462134
90,"Augsburg, Stadt",0.565015
63,Mittelfranken,0.389342
89,Schwaben,0.268312
40,"Regensburg, Stadt",0.135737
66,"Fürth, Stadt",0.112126
76,Unterfranken,0.073585


In [34]:
df_wide["demand_raw"] = (
    df_wide["population"] *
    df_wide["population_density"]
)

df_wide["demand_index"] = (
    (df_wide["demand_raw"] - df_wide["demand_raw"].mean()) /
    df_wide["demand_raw"].std()
)

df_wide[["Name", "demand_index"]].sort_values(
    "demand_index", ascending=False
).head(10)

Indikator,Name,demand_index
3,"München, Landeshauptstadt",9.101477
0,Bayern,3.246182
67,Nürnberg,1.850259
1,Oberbayern,1.462134
90,"Augsburg, Stadt",0.565015
63,Mittelfranken,0.389342
89,Schwaben,0.268312
40,"Regensburg, Stadt",0.135737
66,"Fürth, Stadt",0.112126
76,Unterfranken,0.073585


In [35]:
# zeigt, wie die Raumebene-Spalte bei dir heißt
df.columns

Index(['Kennziffer', 'Name', 'Nordbayern', 'Raumbezug', 'Zeitbezug',
       'Indikator', 'Kuerzel', 'Bereich', 'ID', 'Wert'],
      dtype='object')

In [36]:
df["Raumbezug"].value_counts()

Raumbezug
Kreise               788229
Regierungsbezirke     53747
Bundesländer           8244
Name: count, dtype: int64

In [42]:
df_nb = df[
    (df["Nordbayern"] == 1) &
    (df["Raumbezug"] == "Kreise")
].copy()

In [43]:
df_nb["Raumbezug"].value_counts()
df_nb["Name"].head(10)

210               Amberg
211               Amberg
212               Amberg
213               Amberg
214               Amberg
215               Amberg
216    Regensburg, Stadt
217    Regensburg, Stadt
218    Regensburg, Stadt
219    Regensburg, Stadt
Name: Name, dtype: object

In [44]:
df_nb_agg = (
    df_nb
    .groupby(["Kennziffer", "Name", "Indikator"], as_index=False)
    .agg({"Wert": "mean"})
)

df_nb_agg.head(10)

,Kennziffer,Name,Indikator,Wert
0,09361000,Amberg,(SDG 1) Armut - Altersarmut,3.631875
1,09361000,Amberg,(SDG 1) Armut - Kinderarmut,11.246250
2,09361000,Amberg,(SDG 1) SGB II-/SGB XII-Quote,7.957000
3,09361000,Amberg,(SDG 10) Beschäftigungsquote - Ausländer,43.347692
4,09361000,Amberg,(SDG 10) Einbürgerungen,1.098333
5,09361000,Amberg,(SDG 11) Angebotsmietpreise,7.000000
6,09361000,Amberg,(SDG 11) Fertiggestellte Wohngebäude mit erneu...,49.898750
7,09361000,Amberg,(SDG 11) Flächeninanspruchnahme,35.333750
8,09361000,Amberg,(SDG 11) Flächenneuinanspruchnahme,0.182857
9,09361000,Amberg,(SDG 11) Flächennutzungsintensität,420.341250


In [45]:
df_sel = df_nb_agg[
    df_nb_agg["Indikator"].str.contains(
        "Bevölkerung|Einwohnerdichte|Bodenfläche",
        regex=True,
        na=False
    )
].copy()

df_sel["Indikator"].value_counts()

Indikator
Bevölkerung                                            47
Bevölkerung (mit BBSR-Zensuskorrekturen)               47
Schutzsuchende an Bevölkerung                          47
Prognostizierte Bevölkerungsentwicklung (2022-2045)    47
Prognostizierte Bevölkerungsentwicklung (2022-2040)    47
Prognostizierte Bevölkerungsentwicklung (2022-2035)    47
Prognostizierte Bevölkerungsentwicklung (2022-2030)    47
Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre)      47
Einwohnerdichte                                        47
Durchschnittsalter der Bevölkerung                     47
Bodenfläche gesamt qkm                                 47
Bevölkerungsentwicklung (5 Jahre)                      47
Bevölkerungsentwicklung (10 Jahre)                     47
Bevölkerungsentwicklung                                47
Bevölkerung – Wanderungen                              47
Bevölkerung – Bevölkerungsstruktur                     47
Bevölkerung – Altersstruktur                           47
Bevö

In [46]:
df_wide = (
    df_sel
    .pivot(
        index=["Kennziffer", "Name"],
        columns="Indikator",
        values="Wert"
    )
    .reset_index()
)

df_wide.head()

Indikator,Kennziffer,Name,Bevölkerung,Bevölkerung (mit BBSR-Zensuskorrekturen),Bevölkerung - Natürliche Bevölkerungsbewegungen,Bevölkerung gesamt,Bevölkerung in Mittelzentren,Bevölkerung in Oberzentren,Bevölkerung männlich,Bevölkerung weiblich,Bevölkerung – Altersstruktur,Bevölkerung – Bevölkerungsstruktur,Bevölkerung – Wanderungen,Bevölkerungsentwicklung,Bevölkerungsentwicklung (10 Jahre),Bevölkerungsentwicklung (5 Jahre),Bodenfläche gesamt qkm,Durchschnittsalter der Bevölkerung,Einwohnerdichte,Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre),Prognostizierte Bevölkerungsentwicklung (2022-2030),Prognostizierte Bevölkerungsentwicklung (2022-2035),Prognostizierte Bevölkerungsentwicklung (2022-2040),Prognostizierte Bevölkerungsentwicklung (2022-2045),Schutzsuchende an Bevölkerung,Schutzsuchende an ausländischer Bevölkerung
0,09361000,Amberg,32774.222222,42377.862069,100.00,43044.896552,0.00,100.000,20842.758621,22202.137931,100.00,100.00,100.00,0.564286,-0.926207,-0.551034,50.122308,43.671379,846.657308,28005.034483,-1.66,-2.38,-3.09,-3.80,1.870000,18.348824
1,09362000,"Regensburg, Stadt",117753.666667,135197.689655,100.00,136986.620690,0.00,100.000,65828.827586,71157.793103,100.00,100.00,100.00,4.185714,7.029310,3.521379,80.793462,41.463448,1786.836154,95568.448276,4.42,5.50,6.73,7.81,2.491765,15.272353
2,09363000,Weiden i.d.OPf.,33077.111111,42309.724138,100.00,42522.310345,0.00,100.000,20190.896552,22331.413793,100.00,100.00,100.00,0.908571,0.085517,-0.137241,70.320769,43.440000,602.735769,27636.586207,-0.71,-1.18,-1.42,-1.90,2.085882,19.225294
3,09371000,Amberg-Sulzbach,80357.366667,105091.827586,99.65,105608.793103,18.64,0.000,52444.965517,53163.827586,99.65,99.65,99.65,0.141429,1.845172,0.204138,1255.774615,41.976207,82.906923,69586.620690,-0.29,-0.68,-1.35,-2.22,0.650000,10.841176
4,09372000,Cham,99515.304444,127507.620690,99.37,128822.862069,27.65,13.378,64114.896552,64707.965517,99.37,99.37,99.37,1.122857,1.233448,0.084828,1521.296923,42.103103,84.059231,85143.310345,0.55,0.47,0.08,-0.63,0.636471,11.360588


In [47]:
rename_map = {}

for c in df_wide.columns:
    if isinstance(c, str):
        if "Bevölkerung" in c and "gesamt" in c:
            rename_map[c] = "population"
        elif "Einwohnerdichte" in c:
            rename_map[c] = "population_density"
        elif "Bodenfläche" in c:
            rename_map[c] = "area_km2"

df_wide = df_wide.rename(columns=rename_map)

df_wide.columns

Index(['Kennziffer', 'Name', 'Bevölkerung',
       'Bevölkerung (mit BBSR-Zensuskorrekturen)',
       'Bevölkerung - Natürliche Bevölkerungsbewegungen', 'population',
       'Bevölkerung in Mittelzentren', 'Bevölkerung in Oberzentren',
       'Bevölkerung männlich', 'Bevölkerung weiblich',
       'Bevölkerung – Altersstruktur', 'Bevölkerung – Bevölkerungsstruktur',
       'Bevölkerung – Wanderungen', 'Bevölkerungsentwicklung',
       'Bevölkerungsentwicklung (10 Jahre)',
       'Bevölkerungsentwicklung (5 Jahre)', 'area_km2',
       'Durchschnittsalter der Bevölkerung', 'population_density',
       'Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2030)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2035)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2040)',
       'Prognostizierte Bevölkerungsentwicklung (2022-2045)',
       'Schutzsuchende an Bevölkerung',
       'Schutzsuchende an ausländischer Bevölkerung']

In [48]:
[c for c in df_wide.columns if "Bevölkerung" in c]

['Bevölkerung',
 'Bevölkerung (mit BBSR-Zensuskorrekturen)',
 'Bevölkerung - Natürliche Bevölkerungsbewegungen',
 'Bevölkerung in Mittelzentren',
 'Bevölkerung in Oberzentren',
 'Bevölkerung männlich',
 'Bevölkerung weiblich',
 'Bevölkerung – Altersstruktur',
 'Bevölkerung – Bevölkerungsstruktur',
 'Bevölkerung – Wanderungen',
 'Bevölkerungsentwicklung',
 'Bevölkerungsentwicklung (10 Jahre)',
 'Bevölkerungsentwicklung (5 Jahre)',
 'Durchschnittsalter der Bevölkerung',
 'Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre)',
 'Prognostizierte Bevölkerungsentwicklung (2022-2030)',
 'Prognostizierte Bevölkerungsentwicklung (2022-2035)',
 'Prognostizierte Bevölkerungsentwicklung (2022-2040)',
 'Prognostizierte Bevölkerungsentwicklung (2022-2045)',
 'Schutzsuchende an Bevölkerung',
 'Schutzsuchende an ausländischer Bevölkerung']

In [49]:
df_wide = df_wide.rename(columns={
    "Bevölkerung (mit BBSR-Zensuskorrekturen)": "population",
    "Einwohnerdichte (Einwohner je km²)": "population_density",
    "Bodenfläche": "area_km2"
})

In [50]:
keep_cols = [
    "Kennziffer",
    "Name",
    "population",
    "population_density",
    "area_km2"
]

df_wide_clean = df_wide[keep_cols].copy()
df_wide_clean.head()

Indikator,Kennziffer,Name,population,population,population_density,area_km2
0,09361000,Amberg,42377.862069,43044.896552,846.657308,50.122308
1,09362000,"Regensburg, Stadt",135197.689655,136986.620690,1786.836154,80.793462
2,09363000,Weiden i.d.OPf.,42309.724138,42522.310345,602.735769,70.320769
3,09371000,Amberg-Sulzbach,105091.827586,105608.793103,82.906923,1255.774615
4,09372000,Cham,127507.620690,128822.862069,84.059231,1521.296923


In [52]:
df_wide_clean.columns[df_wide_clean.columns.duplicated()].tolist()

['population']

In [53]:
df_wide_clean = df_wide_clean.loc[:, ~df_wide_clean.columns.duplicated()].copy()

In [54]:
df_wide_clean["demand_raw"] = (
    df_wide_clean["population"].astype(float) *
    df_wide_clean["population_density"].astype(float)
)

df_wide_clean["demand_index"] = (
    (df_wide_clean["demand_raw"] - df_wide_clean["demand_raw"].mean()) /
    df_wide_clean["demand_raw"].std()
)

df_wide_clean.sort_values("demand_index", ascending=False).head(10)

Indikator,Kennziffer,Name,population,population_density,area_km2,demand_raw,demand_index
26,09564000,Nürnberg,495319.482759,2734.330769,186.421154,1.354367e+09,6.448377
1,09362000,"Regensburg, Stadt",135197.689655,1786.836154,80.793462,2.415761e+08,0.871224
25,09563000,"Fürth, Stadt",117828.448276,1946.245385,63.350000,2.293231e+08,0.809813
37,09663000,"Würzburg, Stadt",126462.724138,1464.829231,87.606154,1.852463e+08,0.588906
24,09562000,Erlangen,104809.931034,1420.345769,76.943462,1.488663e+08,0.406575
10,09461000,"Bamberg, Stadt",71124.172414,1354.972308,54.627692,9.637128e+07,0.143477
36,09662000,"Schweinfurt, Stadt",53193.379310,1495.548846,35.695769,7.955330e+07,0.059188
11,09462000,"Bayreuth, Stadt",72406.137931,1095.650000,66.905000,7.933179e+07,0.058078
35,09661000,"Aschaffenburg, Stadt",68274.206897,1117.226923,62.467692,7.627778e+07,0.042771
30,09573000,Fürth,113564.724138,377.517308,307.483077,4.287265e+07,-0.124650


In [57]:
[c for c in df_wide.columns if "Einwohnerdichte" in c]
[c for c in df_wide.columns if "Bodenfläche" in c]
[c for c in df_wide.columns if "Bevölkerung" in c and "Zensuskorrekturen" in c]

[]

In [59]:
# 1) Dichte-Spalten finden (Einwohnerdichte / Bevölkerungsdichte / Dichte)
dens_candidates = [c for c in df_wide.columns if isinstance(c, str) and (
    "dichte" in c.lower() and ("einwohner" in c.lower() or "bevölker" in c.lower())
)]
print("Dichte-Kandidaten:", dens_candidates)

# 2) Flächen-Spalten finden (Bodenfläche / Fläche)
area_candidates = [c for c in df_wide.columns if isinstance(c, str) and (
    "bodenfläche" in c.lower() or (c.lower() == "fläche") or ("fläche" in c.lower() and "boden" in c.lower())
)]
print("Fläche-Kandidaten:", area_candidates)

# 3) Bevölkerungs-Spalten finden (mit Zensuskorrektur bevorzugt)
pop_candidates = [c for c in df_wide.columns if isinstance(c, str) and "bevölker" in c.lower()]
print("Bevölkerung-Kandidaten (Auszug):", pop_candidates[:20])

Dichte-Kandidaten: []
Fläche-Kandidaten: []
Bevölkerung-Kandidaten (Auszug): ['Bevölkerung', 'Bevölkerung - Natürliche Bevölkerungsbewegungen', 'Bevölkerung in Mittelzentren', 'Bevölkerung in Oberzentren', 'Bevölkerung männlich', 'Bevölkerung weiblich', 'Bevölkerung – Altersstruktur', 'Bevölkerung – Bevölkerungsstruktur', 'Bevölkerung – Wanderungen', 'Bevölkerungsentwicklung', 'Bevölkerungsentwicklung (10 Jahre)', 'Bevölkerungsentwicklung (5 Jahre)', 'Durchschnittsalter der Bevölkerung', 'Erwerbsfähige Bevölkerung (15 bis unter 65 Jahre)', 'Prognostizierte Bevölkerungsentwicklung (2022-2030)', 'Prognostizierte Bevölkerungsentwicklung (2022-2035)', 'Prognostizierte Bevölkerungsentwicklung (2022-2040)', 'Prognostizierte Bevölkerungsentwicklung (2022-2045)', 'Schutzsuchende an Bevölkerung', 'Schutzsuchende an ausländischer Bevölkerung']
